In [317]:
#Brianna Sanchez and Alexander Zhuk

In [318]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegressionCV
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

In [319]:
data = pd.read_csv('shopping_behavior_updated.csv')

In [320]:
data.head(5)

,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,Yes,49,PayPal,Weekly
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,Yes,31,PayPal,Annually


In [321]:
print("Dataset shape:", data.shape)

Dataset shape: (3900, 18)


In [322]:
print("Number of records:", len(data))

Number of records: 3900


In [323]:
print("Number of variables:", len(data.columns))

Number of variables: 18


In [324]:
print("Variable names:", data.columns.tolist())

Variable names: ['Customer ID', 'Age', 'Gender', 'Item Purchased', 'Category', 'Purchase Amount (USD)', 'Location', 'Size', 'Color', 'Season', 'Review Rating', 'Subscription Status', 'Shipping Type', 'Discount Applied', 'Promo Code Used', 'Previous Purchases', 'Payment Method', 'Frequency of Purchases']


In [325]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3900 entries, 0 to 3899
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Customer ID             3900 non-null   int64  
 1   Age                     3900 non-null   int64  
 2   Gender                  3900 non-null   object 
 3   Item Purchased          3900 non-null   object 
 4   Category                3900 non-null   object 
 5   Purchase Amount (USD)   3900 non-null   int64  
 6   Location                3900 non-null   object 
 7   Size                    3900 non-null   object 
 8   Color                   3900 non-null   object 
 9   Season                  3900 non-null   object 
 10  Review Rating           3900 non-null   float64
 11  Subscription Status     3900 non-null   object 
 12  Shipping Type           3900 non-null   object 
 13  Discount Applied        3900 non-null   object 
 14  Promo Code Used         3900 non-null   

In [326]:
print(data.describe())

       Customer ID          Age  ...  Review Rating  Previous Purchases
count  3900.000000  3900.000000  ...    3900.000000         3900.000000
mean   1950.500000    44.068462  ...       3.749949           25.351538
std    1125.977353    15.207589  ...       0.716223           14.447125
min       1.000000    18.000000  ...       2.500000            1.000000
25%     975.750000    31.000000  ...       3.100000           13.000000
50%    1950.500000    44.000000  ...       3.700000           25.000000
75%    2925.250000    57.000000  ...       4.400000           38.000000
max    3900.000000    70.000000  ...       5.000000           50.000000

[8 rows x 5 columns]


In [327]:
print(data.isnull().values.any()) #nulls
print(data.duplicated().sum()) #duplicates
data.columns = data.columns.str.strip().str.lower().str.replace(" ", "_") #python language to remove spaces with _

False
0


In [328]:
#Feature Engineering
#Dropping irrelevant columns, encoding categorical variables
data = data.drop(columns=['location','item_purchased','color','customer_id'])
data = pd.get_dummies(data, columns=['gender','category','size','season','subscription_status', 'discount_applied',
                'promo_code_used', 'payment_method',])
freq_mapping = {
    'Every 3 Months': 'Quarterly',
    'Bi-Weekly': 'Fortnightly'
}
data['frequency_of_purchases'] = data['frequency_of_purchases'].replace(freq_mapping)
freq_order = {
    'Weekly': 6,
    'Fortnightly': 5,
    'Monthly': 4,
    'Quarterly': 3,
    'Annually': 2
}
data['frequency_numeric'] = data['frequency_of_purchases'].map(freq_order)

data = data.drop(columns=['frequency_of_purchases'])

In [329]:
data.head(10)

,age,purchase_amount_(usd),review_rating,shipping_type,previous_purchases,gender_Female,gender_Male,category_Accessories,category_Clothing,category_Footwear,category_Outerwear,size_L,size_M,size_S,size_XL,season_Fall,season_Spring,season_Summer,season_Winter,subscription_status_No,subscription_status_Yes,discount_applied_No,discount_applied_Yes,promo_code_used_No,promo_code_used_Yes,payment_method_Bank Transfer,payment_method_Cash,payment_method_Credit Card,payment_method_Debit Card,payment_method_PayPal,payment_method_Venmo,frequency_numeric
0,55,53,3.1,Express,14,False,True,False,True,False,False,True,False,False,False,False,False,False,True,False,True,False,True,False,True,False,False,False,False,False,True,5
1,19,64,3.1,Express,2,False,True,False,True,False,False,True,False,False,False,False,False,False,True,False,True,False,True,False,True,False,True,False,False,False,False,5
2,50,73,3.1,Free Shipping,23,False,True,False,True,False,False,False,False,True,False,False,True,False,False,False,True,False,True,False,True,False,False,True,False,False,False,6
3,21,90,3.5,Next Day Air,49,False,True,False,False,True,False,False,True,False,False,False,True,False,False,False,True,False,True,False,True,False,False,False,False,True,False,6
4,45,49,2.7,Free Shipping,31,False,True,False,True,False,False,False,True,False,False,False,True,False,False,False,True,False,True,False,True,False,False,False,False,True,False,2
5,46,20,2.9,Standard,14,False,True,False,False,True,False,False,True,False,False,False,False,True,False,False,True,False,True,False,True,False,False,False,False,False,True,6
6,63,85,3.2,Free Shipping,49,False,True,False,True,False,False,False,True,False,False,True,False,False,False,False,True,False,True,False,True,False,True,False,False,False,False,3
7,27,34,3.2,Free Shipping,19,False,True,False,True,False,False,True,False,False,False,False,False,False,True,False,True,False,True,False,True,False,False,True,False,False,False,6
8,26,97,2.6,Express,8,False,True,False,False,False,True,True,False,False,False,False,False,True,False,False,True,False,True,False,True,False,False,False,False,False,True,2
9,57,31,4.8,2-Day Shipping,4,False,True,True,False,False,False,False,True,False,False,False,True,False,False,False,True,False,True,False,True,False,True,False,False,False,False,3


In [330]:
#Partitioning data 60% training and 40% validation
X = data.drop(columns=['shipping_type', 'purchase_amount_(usd)'])
# Dropping column Location in X
y = data['purchase_amount_(usd)']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.4, random_state = 42)
#Performed split for training and test sets
scaler = StandardScaler()
#Created a StandardScaler object
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
# Scaling only numeric features
print(X_train.shape, y_train.shape)

(2340, 30) (2340,)


In [331]:
 logit_full = LogisticRegressionCV(penalty="l2", cv=5,solver='lbfgs',max_iter=500, class_weight='balanced')
#Created cross-validated logistic regression with balanced classes
#Applid L2 regularization (Ridge)
#Applied 5-fold cross-validation
#Used lbfgs to find model coeficients
#Set maximum number of iterations to 500
logit_full.fit(X_train_scaled, y_train)
#Trained logistic regression model on scaled data

LogisticRegressionCV(class_weight='balanced', cv=5, max_iter=500)

In [332]:
#Evaluating which variables impact purchase amount the most
pd.set_option('display.width', 95)
pd.set_option('display.precision',3)
pd.set_option('display.max_columns', 33)
print('intercept ', logit_full.intercept_[0], '\n')
# could use display() to present this cleaner
#Set pandas display options and printed logistic regression intercept
print(pd.DataFrame({'coeff': logit_full.coef_[0]}, index=X.columns).transpose())
pd.reset_option('display.width')
pd.reset_option('display.precision')
pd.reset_option('display.max_columns')
#Displayed model coefficients in DataFrame, reset display options

intercept  0.005037702833550866 

         age  review_rating  previous_purchases  gender_Female  gender_Male  \
coeff  0.002          0.015               0.023         -0.026        0.026   

       category_Accessories  category_Clothing  category_Footwear  category_Outerwear  \
coeff                -0.015              0.015              0.009              -0.014   

       size_L  size_M  size_S  size_XL  season_Fall  season_Spring  season_Summer  \
coeff  -0.029  -0.012   0.035    0.017       -0.051          0.022          0.025   

       season_Winter  subscription_status_No  subscription_status_Yes  discount_applied_No  \
coeff          0.003                  -0.017                    0.017                0.008   

       discount_applied_Yes  promo_code_used_No  promo_code_used_Yes  \
coeff                -0.008               0.008               -0.008   

       payment_method_Bank Transfer  payment_method_Cash  payment_method_Credit Card  \
coeff                         -0.04